# Defensive Sync Score (DSS)
**NFL Big Data Bowl 2021 — Tracking Data Analysis**

A new metric quantifying how synchronized a defensive secondary moves as a unit.

**Three sub-scores (post-snap, snap → pass release):**
- **VA — Velocity Alignment (40%):** Do DBs move in the same direction? (cosine similarity of velocity vectors)
- **SD — Spacing Discipline (30%):** Do they maintain their formation? (inverse of inter-DB distance CV)
- **OD — Orientation Divergence (30%):** Is each DB tracking his own assignment? (inverted circular variance of body orientation)

**DSS = 100 × (0.40·VA + 0.30·SD + 0.30·OD)**

In [ ]:
import sys, os
from pathlib import Path

# Adjust path to find src/
sys.path.insert(0, str(Path(os.getcwd()) / 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
from IPython.display import display, HTML

from dss import load_metadata, load_tracking_data, identify_secondary, compute_dss_all_plays
from dss_visualization import plot_dss_team_ranking, plot_dss_vs_outcome, plot_dss_heatmap
from dashboard import build_dashboard

DATA_DIR = Path('data')
OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)

print('Setup complete.')

## 1. Load Data

In [ ]:
meta = load_metadata(DATA_DIR)
games, plays, players, scouting = meta['games'], meta['plays'], meta['players'], meta['scouting']

print(f'Games: {len(games)} | Plays: {len(plays)} | Players: {len(players)} | Scouting rows: {len(scouting)}')
plays[['gameId','playId','possessionTeam','defensiveTeam','passResult','pff_passCoverage','pff_passCoverageType']].head()

In [ ]:
print('Loading tracking data (122 games × ~68k rows each)...')
tracking = load_tracking_data(DATA_DIR / 'tracking')
tracking = tracking.merge(
    players[['nflId','officialPosition']].rename(columns={'officialPosition':'position'}),
    on='nflId', how='left',
)
print(f'Total tracking rows: {len(tracking):,}')
tracking.head(3)

## 2. Compute DSS for All Plays

In [ ]:
dss_df = compute_dss_all_plays(tracking, scouting, players, plays, games)

print(f'Plays with DSS: {len(dss_df)}')
print(f'DSS range: {dss_df["dss"].min():.1f} — {dss_df["dss"].max():.1f}')
print(f'DSS mean: {dss_df["dss"].mean():.1f}')
dss_df.head()

## 3. Statistical Validation

In [ ]:
# Does higher DSS → fewer completions?
hi = dss_df[dss_df['dss'] >= dss_df['dss'].median()]
lo = dss_df[dss_df['dss'] < dss_df['dss'].median()]

print('=== DSS vs Completion Rate ===')
print(f'High DSS: {hi["completed"].mean():.3f} completion rate')
print(f'Low DSS:  {lo["completed"].mean():.3f} completion rate')
print(f'Difference: {(hi["completed"].mean() - lo["completed"].mean())*100:+.1f} pp')
print()

for ct in ['Man', 'Zone']:
    sub = dss_df[dss_df['coverageType'] == ct]
    hi2 = sub[sub['dss'] >= sub['dss'].median()]
    lo2 = sub[sub['dss'] < sub['dss'].median()]
    diff = (hi2['completed'].mean() - lo2['completed'].mean()) * 100
    print(f'{ct}: high DSS={hi2["completed"].mean():.3f}, low DSS={lo2["completed"].mean():.3f}, delta={diff:+.1f}pp (n={len(sub)})')

print()
print(f'Pearson(DSS, completed): {dss_df["dss"].corr(dss_df["completed"]):.4f}')

## 4. Visualizations

In [ ]:
# Team ranking
plot_dss_team_ranking(dss_df, output_path=str(OUTPUT_DIR / 'dss_team_ranking.png'))
from IPython.display import Image
Image(str(OUTPUT_DIR / 'dss_team_ranking.png'), width=700)

In [ ]:
# DSS vs Completion
plot_dss_vs_outcome(dss_df, output_path=str(OUTPUT_DIR / 'dss_vs_completion.png'))
Image(str(OUTPUT_DIR / 'dss_vs_completion.png'), width=800)

In [ ]:
# Heatmap
plot_dss_heatmap(dss_df, output_path=str(OUTPUT_DIR / 'dss_heatmap.png'))
Image(str(OUTPUT_DIR / 'dss_heatmap.png'), width=700)

In [ ]:
# Interactive dashboard
dash_path = build_dashboard(
    str(OUTPUT_DIR / 'dss_results.csv'),
    str(OUTPUT_DIR / 'dss_dashboard.html'),
)
print(f'Dashboard: {dash_path}')
display(HTML(f'<a href="{dash_path}" target="_blank">Open interactive dashboard</a>'))

## 5. Key Findings

| Finding | Value |
|---|---|
| Best synchronized secondary | **SEA** (DSS = 59.0) |
| Least synchronized | **LAC** (DSS = 51.5) |
| Man coverage delta (high vs low DSS) | **-6.1 pp** completion rate |
| Zone coverage delta | **-3.7 pp** completion rate |
| Hardest coverage to sync | Cover-0 / Cover-1 (~44–47) |
| Easiest to sync | Prevent / Quarters (~60–63) |
| Counterintuitive insight | Orientation **divergence** (each DB tracking own man) > coherence (all staring at QB) |

### Interpretation
The strongest signal is in **man coverage**: when DBs move together and each one tracks their own assignment (rather than all staring at the quarterback), completions drop by 6 percentage points. This is actionable for defensive coordinators — DSS identifies breakdowns in communication that happen before the ball is even thrown.